# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets, fields, and their `@id` values. All references are managed by their `@id` for consistency.

In [ ]:
# List record sets, fields and columns
record_sets = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]

print("Available Record Sets (@id):")
for rs_id in record_sets:
    print(f"- {rs_id}")

fields_map = {}
for rs_id in record_sets:
    rs_json = None
    for rs in dataset.metadata.to_json()['recordSet']:
        if rs['@id'] == rs_id:
            rs_json = rs
            break
    if not rs_json:
        continue
    print(f"\nFields and columns in Record Set {rs_id}:")
    if 'field' in rs_json:
        if isinstance(rs_json['field'], dict):
            fields = [rs_json['field']]
        else:
            fields = rs_json['field']
        for f in fields:
            print(f"  - Field @id: {f['@id']}, Name: {f.get('name', '')}, DataType: {f.get('dataType', '')}")
            fields_map[f['@id']] = f.get('name', '')

## 3. Data Extraction
Load records from a specific record set as a pandas DataFrame. Reference the record set and field by their `@id`.

In [ ]:
# Choose the main tabular record set for extraction
main_record_sets = record_sets if record_sets else []

dataframes = {}
for rs_id in main_record_sets:
    print(f"\nLoading records for RecordSet {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Columns (@id): {dataframes[rs_id].columns.tolist()}")
    print(dataframes[rs_id].head())

# For further analysis, pick the first RecordSet
if main_record_sets:
    record_set_id = main_record_sets[0]
else:
    record_set_id = None
df = dataframes.get(record_set_id, pd.DataFrame())

## 4. Exploratory Data Analysis (EDA)
Apply typical preprocessing steps, referencing fields and columns by their `@id`.

In [ ]:
# Identify numeric fields for analysis
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Example: use first numeric column
    print(f"Using numeric field: {numeric_field_id}")

    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > mean:")
    print(filtered_df.head())

    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Try grouping by a categorical field (e.g. anatomical location, sex)
    group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
    group_field_id = group_fields[0] if group_fields else None
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped results by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields. Reference plot axes by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_fields:
    # Distribution of main numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.show()

    # Relationship with a categorical field
    if group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored a clinical dataset using `mlcroissant`, referencing all entities by their `@id`. We examined record sets, extracted tabular data, performed basic filtering and normalization on numeric fields, grouped by key categorical attributes, and visualized data distributions. This workflow paves the way for deeper clinical and biomarker analysis of second primary colorectal cancer in cancer survivors.